# 04 — Paired patch extraction and reconstruction

`rp.extract_patches` slides a window over each reference slide and saves the
same window from its aligned counterpart wherever the reference holds enough
tissue — pixel-matched H&E/IHC pairs, e.g. for training. A metadata file
records every patch position so (processed) patches can be stitched back into
a slide.

It takes the output of `rp.align` directly; no copying or renaming is needed.

In [ ]:
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository whether Jupyter starts at its root or in how_to_use/."""
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "rocqipath").is_dir():
            return candidate
    return here


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data"          # your slides (kept out of git)
RESULTS_ROOT = PROJECT_ROOT / "results"    # outputs (kept out of git)
DEMO_ROOT = PROJECT_ROOT / "notebook_demo_outputs"  # synthetic examples

import rocqipath as rp

print(f"RocqiPath {rp.__version__}")
print(f"Data    : {DATA_ROOT}")
print(f"Results : {RESULTS_ROOT}")

In [ ]:
ALIGNED = RESULTS_ROOT / "aligned"     # output folder of notebook 03
OUTPUT_DIR = RESULTS_ROOT / "patches"

patch_settings = dict(
    patch_size=512,
    stride=512,                 # smaller than patch_size = overlapping patches
    tissue_threshold=0.50,
    target_magnification=20.0,
    moving_name="cd8",          # stain label used in patch filenames
    max_workers=4,
)

RUN_PATCH_EXTRACTION = False

In [ ]:
if RUN_PATCH_EXTRACTION:
    patches = rp.extract_patches(ALIGNED, OUTPUT_DIR, **patch_settings)
    print(f"Processed: {patches.summary['processed']}  skipped: {patches.summary['skipped']}")
    he = [item for item in patches if item.meta["stain"] == "he"]
    print(f"H&E patches: {len(he)}; first: {he[0].path.name if he else '-'}")
else:
    print("Set RUN_PATCH_EXTRACTION=True after running notebook 03.")

## The case folder

```text
results/patches/patch_extraction/Sample_0001_CD8/
├── Sample_0001_CD8_he_patch_000001.png
├── Sample_0001_CD8_cd8_patch_000001.png
└── Sample_0001_CD8_metadata.json        # positions: the authoritative pairing
```

In [ ]:
import json

manifests = sorted((OUTPUT_DIR / "patch_extraction").glob("*/*_metadata.json"))
print(f"Case manifests: {len(manifests)}")
if manifests:
    metadata = json.loads(manifests[0].read_text())
    print(metadata["case_id"], metadata["dimensions"], f"{len(metadata['patches'])} patches")
    from rocqipath.viz import view_pairs
    view_pairs(str(manifests[0].parent), num_to_show=min(5, len(metadata["patches"])))

## Reconstruct a slide from processed patches

Put one predicted image per patch ID in
`patch_extraction/<case>/predicted_ihc/`, then rebuild a pyramidal TIFF with
`ReversiblePatchExtractor.reconstruct_wsi(..., mode="predicted_ihc")`.
Overlapping patches (stride < patch size) are averaged.

In [ ]:
from rocqipath.extraction import ReversiblePatchExtractor

RUN_RECONSTRUCTION = False
CASE_ID = "Sample_0001_CD8"

if RUN_RECONSTRUCTION:
    reconstructor = ReversiblePatchExtractor({
        "he_root": str(DATA_ROOT / "pairs"),
        "aligned_root": str(ALIGNED),
        "biomarker_folders": ["CD8"],
        "output_dir": str(OUTPUT_DIR),
        "patch_size": patch_settings["patch_size"],
    })
    print(reconstructor.reconstruct_wsi(CASE_ID, "CD8", str(OUTPUT_DIR), mode="predicted_ihc"))
else:
    print("Set RUN_RECONSTRUCTION=True once predictions exist.")

**Choices** — `stride == patch_size` gives independent patches and simple
reconstruction; a lower `tissue_threshold` keeps edge patches; `max_workers`
parallelizes cases, not patches.